In [ ]:
import os
from glob import glob
from tqdm import tqdm
import datetime
import hail as hl

from hail.plot import show
import pandas as pd
from pprint import pprint
hl.plot.output_notebook()

from pyspark.conf import SparkConf
from pyspark.context import SparkContext
from pyspark.sql.session import SparkSession

#### Start an [Apache Spark](https://en.wikipedia.org/wiki/Apache_Spark) instance

In [ ]:
# TODO: set external ip
import socket

hostname = socket.gethostname()
print(f"hostname: {hostname}")
internal_ip = socket.gethostbyname(hostname)
#external_ip = "172.19.179.106"
# print(f"internal ip: {internal_ip}")
# print(f"external ip: {external_ip}")

In [ ]:
HAIL_JARS = hl.__path__[0]
HAIL_JARS += "/backend/hail-all-spark.jar"
HAIL_JARS += ",/spark/jars/aws-java-sdk-bundle-1.11.1026.jar"
HAIL_JARS += ",/spark/jars/hadoop-aws-3.3.2.jar"
print(HAIL_JARS)

In [ ]:
log_file_name = f"logs/hail-{datetime.datetime.now():%Y-%m-%d-%H-%M-%S}.log"
# run spark
spark_conf = SparkConf().setAppName("hail-test").setMaster("spark://172.19.179.106:30077")
# hail
spark_conf.set("spark.jars", HAIL_JARS)
spark_conf.set("spark.driver.host", internal_ip)
spark_conf.set("spark.driver.bindAddress", internal_ip)
spark_conf.set("spark.driver.port", 32123)
spark_conf.set("spark.blockManager.port", 32124)
# s3
spark_conf.set("spark.hadoop.fs.s3a.endpoint", "http://172.19.179.106:30900/")
spark_conf.set("spark.hadoop.fs.s3a.access.key", "root")
spark_conf.set("spark.hadoop.fs.s3a.secret.key", "passpass" )
spark_conf.set("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
spark_conf.set("spark.hadoop.fs.s3a.path.style.access", "true")
spark_conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
spark_conf.set("spark.hadoop.fs.s3a.connection.maximum", 1024);
spark_conf.set("spark.hadoop.fs.s3a.threads.max", 1024);
spark_conf.set("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
# s3 timeout
spark_conf.set("spark.hadoop.fs.s3a.connection.timeout", 60000)          # default: 5000 (ms)
spark_conf.set("spark.hadoop.fs.s3a.socket.timeout" , 120000)            # default: 5000 (ms)
spark_conf.set("spark.hadoop.fs.s3a.attempts.maximum", 20)               # default: 10
spark_conf.set("spark.hadoop.fs.s3a.connection.maximum", 100)            # max simultaneous connections
spark_conf.set("spark.hadoop.fs.s3a.retry.limit", 15)                    # retries per operation
spark_conf.set("spark.hadoop.fs.s3a.retry.interval", 2000)               # wait between retries in ms
spark_conf.set("spark.hadoop.fs.s3a.impl.retry.policy", "exponential")
spark_conf.set("spark.hadoop.fs.s3a.retry.policy.enabled", "true")
# varia
spark_conf.set("spark.executor.memory", "256g")
spark_conf.set("spark.driver.memory", "200g")
spark_conf.set("spark.driver.maxResultSize", "128g")
spark_conf.set("spark.dynamicAllocation.enabled", "true")
spark_conf.set("spark.worker.cleanup.enabled", "false");
spark_conf.set("spark.worker.cleanup.interval", 600);
spark_conf.set("spark.worker.cleanup.appDataTtl", 600);
spark_conf.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
spark_conf.set("spark.kryoserializer.buffer.max", "2047m")
spark_conf.set("spark.rpc.message.maxSize", "512")
spark_conf.set("spark.network.timeout", "600s")
spark_conf.set("spark.dynamicAllocation.maxExecutors", "15")

try:
    sc = SparkContext(conf=spark_conf)
except:
    print ("Spark session already up")

#### Create bucket on [Minio](https://min.io/) if it does not exists

In [ ]:
#bucket_name = "data-hail"
bucket_name = "lifemap"
tmp_bucket = "tmp"
tmp_dir = f's3://{tmp_bucket}/'

### [Hail](https://hail.is/) initialization

In [ ]:
hl.init(sc=sc, log=log_file_name, tmp_dir=tmp_dir)

#### Set filenames

In [ ]:
# Matrix table
fn = f's3://{bucket_name}/gnomad.genomes.v3.1.2/hgdp_1kg_subset_dense_1536p.mt'

In [ ]:
%%time
mt = hl.read_matrix_table(fn)

In [ ]:
%%time
n_variants = mt.count_rows() # This can be done accessing directy the row table with mt.rows().count()
n_samples = mt.count_cols()  # This can be done accessing directy the column table with mt.cols().count()

print (f"\n\nTable has {n_variants} variants and {n_samples} samples") 

In [ ]:
%%time
# sample_qc is a hail genetic method to compute per-sample metrics useful for quality control.
mt = hl.sample_qc(mt)
mt = mt.filter_cols((mt.sample_qc.dp_stats.mean >= 4) & (mt.sample_qc.call_rate >= 0.97))
print('After filter, %d/284 samples remain.' % mt.count_cols())

In [ ]:
%%time
ab = mt.AD[1] / hl.sum(mt.AD)

filter_condition_ab = ((mt.GT.is_hom_ref() & (ab <= 0.1)) |
                        (mt.GT.is_het() & (ab >= 0.25) & (ab <= 0.75)) |
                        (mt.GT.is_hom_var() & (ab >= 0.9)))

fraction_filtered = mt.aggregate_entries(hl.agg.fraction(~filter_condition_ab))
print(f'Filtering {fraction_filtered * 100:.2f}% entries out of downstream analysis.')
mt = mt.filter_entries(filter_condition_ab)

Variant QC computes per per-variant metric useful for quality control. It is a bit more of the same of sample_qc: we can use the [`variant_qc`](https://hail.is/docs/0.2/methods/genetics.html#hail.methods.variant_qc) function to produce a variety of useful statistics, plot them, and filter. This is made at row level beacause they are stats on variants.

In [ ]:
%%time
mt = hl.variant_qc(mt)

In [ ]:
%%time
mt = mt.filter_rows(mt.variant_qc.AF[1] > 0.01) # It takes variants for which the alternate allele has a frequency larger than 1%
print('Samples: %d  Variants: %d' % (mt.count_cols(), mt.count_rows()))

In [ ]:
%%time
mt = mt.filter_rows(mt.variant_qc.p_value_hwe > 1e-6) # Hardy-Weinberg equilibrium pvalue cut-off
print('Samples: %d  Variants: %d' % (mt.count_cols(), mt.count_rows()))

In [ ]:
%%time
eigenvalues, pcs, _ = hl.hwe_normalized_pca(mt.GT)

In [ ]:
%%time
mt = mt.annotate_cols(scores = pcs[mt.s].scores)

In [ ]:
%%time
p = hl.plot.scatter(mt.scores[0],
                    mt.scores[1],
                    title='PCA', xlabel='PC1', ylabel='PC2')
show(p)